# NOTE: This notebook uses local fallback paths for exploratory analysis.
# Production pipeline runs via run_pipeline.py or the Airflow DAG.

# Stage 2 — Validate: Data Quality Gates (Bronze → Silver)

This notebook walks through the **quality gate checks** applied to both the census block and ICF facility data.

Quality gates enforce:
- Required columns present
- Population bounds (0 ≤ pop ≤ 50,000)
- Bed count > 0
- Null percentage < 5%
- No duplicate GEOIDs
- Valid geometries
- Correct CRS (EPSG:26985)

In [7]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

# No pipeline imports — standalone EDA notebook
Blocks_DC = "../data/intermediate_files/blocks_Washington_DC.shp"
ICF   = "../data/intermediate_files/ICFs_DC.shp"
CRS = "EPSG:26985"

# Load raw data directly from shapefiles
blocks = gpd.read_file(Blocks_DC).to_crs(CRS)
facs   = gpd.read_file(ICF).to_crs(CRS)

# Normalise column names to match analysis expectations
pop_raw = blocks.rename(columns={"GEOCODE": "GEOID", "Total Popu": "population"}).copy()
pop_raw["population"] = pd.to_numeric(pop_raw["population"], errors="coerce")

fac_raw = facs.rename(columns={"NAME": "FAC_NAME", "BEDS": "CRTFD_BED_CNT"}).copy()
fac_raw["CRTFD_BED_CNT"] = pd.to_numeric(fac_raw["CRTFD_BED_CNT"], errors="coerce")

print(f"Census blocks (raw)  : {len(pop_raw):,} rows")
print(f"ICF facilities (raw) : {len(fac_raw)} rows")

Census blocks (raw)  : 6,012 rows
ICF facilities (raw) : 114 rows


## 1. Pre-validation Checks — Population

In [8]:
print("=== Population Data ===")
print(f"Null counts:\n{pop_raw.isnull().sum()}\n")
print(f"Population range: [{pop_raw['population'].min()}, {pop_raw['population'].max()}]")
print(f"Invalid geometries: {(~pop_raw.geometry.is_valid).sum()}")
print(f"CRS: {pop_raw.crs}")

=== Population Data ===
Null counts:
OBJECTID      0
BLKGRP        0
BLOCK         0
GEOID_left    0
Bl_totpop     0
TRACT         0
GEOID         0
STATE         0
Longitude     0
Latitude      0
population    0
Total Po_1    0
Total Po_2    0
Total po_3    0
eighteento    0
65 years a    0
TotalPopul    0
Bl_incomep    3
Bl_healthi    3
pop_weight    0
PerCapInc     0
HI_blk        0
age_18_65     0
accessibil    0
geometry      0
dtype: int64

Population range: [13, 7730]
Invalid geometries: 0
CRS: EPSG:26985


## 2. Pre-validation Checks — Facilities

In [5]:
print("=== ICF Facilities ===")
print(f"Null counts:\n{fac_raw.isnull().sum()}\n")
print(f"Beds <= 0: {(fac_raw['CRTFD_BED_CNT'] <= 0).sum()}")
print(f"Bed count range: [{fac_raw['CRTFD_BED_CNT'].min()}, {fac_raw['CRTFD_BED_CNT'].max()}]")
print(f"Invalid geometries: {(~fac_raw.geometry.is_valid).sum()}")
print(f"CRS: {fac_raw.crs}")

=== ICF Facilities ===
Null counts:
FAC_NAME           0
ADDRESS            1
DIRECTOR           0
PHONE              0
CRTFD_BED_CNT      0
LICENSE            0
ID                 0
OBJECTID           0
MAR_ID             0
GIS_ID             0
GLOBALID           0
CREATOR          114
CREATED          114
EDITOR             1
EDITED             1
XCOORD             1
YCOORD             1
LATITUDE           1
LONGITUDE          1
geometry           0
dtype: int64

Beds <= 0: 0
Bed count range: [4, 45]
Invalid geometries: 0
CRS: EPSG:26985


## 3. Quality checks

In [9]:
# Population: drop nulls, fix invalid geometries
pop_clean = pop_raw.copy()
pop_clean = pop_clean[pop_clean["population"].notna()]
pop_clean = pop_clean[pop_clean["population"] >= 0]
pop_clean = pop_clean[~pop_clean["GEOID"].duplicated(keep="first")]
pop_clean = pop_clean[pop_clean.geometry.is_valid]

# Facilities: drop rows with missing/zero beds, fix invalid geometries
fac_clean = fac_raw.copy()
fac_clean = fac_clean[fac_clean["CRTFD_BED_CNT"].notna()]
fac_clean = fac_clean[fac_clean["CRTFD_BED_CNT"] > 0]
fac_clean = fac_clean[fac_clean.geometry.is_valid]

print(f"Population: {len(pop_raw):,} → {len(pop_clean):,} rows (dropped {len(pop_raw)-len(pop_clean)})")
print(f"Facilities: {len(fac_raw)} → {len(fac_clean)} rows (dropped {len(fac_raw)-len(fac_clean)})")

Population: 6,012 → 6,012 rows (dropped 0)
Facilities: 114 → 114 rows (dropped 0)


## 5. Validation Report

In [11]:
report = pd.DataFrame([
    {"Dataset": "Census Blocks", "Before": len(pop_raw), "After": len(pop_clean),
     "Dropped": len(pop_raw)-len(pop_clean),
     "Nulls": int(pop_clean["population"].isnull().sum()),
     "Invalid Geom": int((~pop_clean.geometry.is_valid).sum())},
    {"Dataset": "ICF Facilities", "Before": len(fac_raw), "After": len(fac_clean),
     "Dropped": len(fac_raw)-len(fac_clean),
     "Nulls": int(fac_clean["CRTFD_BED_CNT"].isnull().sum()),
     "Invalid Geom": int((~fac_clean.geometry.is_valid).sum())},
])
report

,Dataset,Before,After,Dropped,Nulls,Invalid Geom
0,Census Blocks,6012,6012,0,0,0
1,ICF Facilities,114,114,0,0,0
